# DaKanjiRecognizer - Single Kanji CNN : ONNX Export

## Setup

In [ ]:
%pip install tf2onnx onnxruntime numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of onnx to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation:

## Download Model Weights

In [ ]:
zip_url = "https://github.com/CaptainDario/DaKanji-Single-Kanji-Recognition/releases/download/v1.2/v1.2.zip" # @param {"type":"string"}
name = zip_url.rsplit('/', 1)[-1].split(".zip")[0]
download_loc = f"/content/{name}.zip"

!wget {zip_url}
!unzip {download_loc}

import os
import shutil
for file in os.listdir(f"/content/{name}/tflite"):
    if file.endswith(".tflite"):
      os.rename(f"/content/{name}/tflite/{file}", f"/content/{file}")
      break
shutil.rmtree(f"/content/{name}")
os.remove(download_loc)


--2025-12-04 02:07:15--  https://github.com/CaptainDario/DaKanji-Single-Kanji-Recognition/releases/download/v1.2/v1.2.zip
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/336364411/f8f42657-9c24-4237-a05b-81bfd7354a46?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-04T02%3A47%3A40Z&rscd=attachment%3B+filename%3Dv1.2.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-04T01%3A47%3A16Z&ske=2025-12-04T02%3A47%3A40Z&sks=b&skv=2018-11-09&sig=5LZ9%2FbuvF%2B3e3H8%2FtMxa2l4jk%2BhpARwvJhjzOAa8pRg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NDgxNTgzNSwibmJmIjoxNzY0ODE0MDM1LCJwYXRoIjoicmVsZWFzZWFzc2V0

## Export

In [ ]:
tflite_loc = "model.tflite" # @param {"type":"string","placeholder":"model.tflite"}
onnx_loc =  "model.onnx" # @param {"type": "string"}
onnx_quant_loc = "model.quant.onnx" # @param {"type": "string"}
opset = 17 # @param {"type":"slider","min":14,"max":17,"step":1}

!python -m tf2onnx.convert --opset {opset} --tflite {tflite_loc} --output model.onnx

# Quantize
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

model_fp32 = "model-infer.onnx"
!python -m onnxruntime.quantization.preprocess --input model.onnx --output model-infer.onnx
quantized_model = quantize_dynamic(model_fp32, onnx_quant_loc, op_types_to_quantize=['MatMul', 'Attention', 'LSTM', 'Gather', 'Transpose', 'EmbedLayerNormalization'])

import os
os.remove(model_fp32)

2025-12-04 02:07:25.811973: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764814045.836821     706 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764814045.844151     706 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764814045.862547     706 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764814045.862608     706 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764814045.862614     706 computation_placer.cc:177] computation placer alr